# Notebook 05: Quality Assessment Visualization

This notebook visualizes the output of the Quality Assessment (QA) framework
produced by notebook `04_qa_framework`. It loads the QA NetCDF files for each
bias correction method (LS, LSEQM, LSEQM+DL) and creates a suite of
diagnostic plots to compare quality across methods and spatial domains.

## Visualizations provided

1. **CQI Spatial Maps** -- Continuous Quality Index for each method side by side.
2. **Categorical Quality Maps** -- Poor / Fair / Good / Excellent classification.
3. **Method Improvement Map** -- Quality gain from LS to LSEQM+DL.
4. **Component Quality Maps** -- Basic statistical, distribution, and temporal scores.
5. **Confidence Map** -- Spatial reliability of the quality assessment.
6. **CQI Distribution Analysis** -- Empirical CDF and histograms comparing methods.
7. **Quality Category Summary** -- Grouped bar chart and percentage table.
8. **Component Score Box Plots** -- Box plots comparing components across methods.

## 1.1 Connect Google Drive (Colab only)

This section is only required when running in **Google Colab** and your project/data are stored in Google Drive.

- Mounting Drive makes your repository and datasets accessible under `/content/drive`.
- If you run this notebook locally (Jupyter / VS Code), **skip this section**.

**Expected structure (Drive)**
After mounting, your project root should contain:
- `notebooks/`
- `src/`
- `config.yml` (or `config.yaml`)

Proceed to the code cell below to mount Drive.

In [ ]:
from google.colab import drive
import os

# Check if the drive is mounted
if os.path.exists("/content/drive"):
    # Try to unmount
    try:
        drive.flush_and_unmount()
        print("Successfully unmounted")
    except:
        print("Unmount failed, the drive might not be mounted or busy")

# Mount the drive
drive.mount("/content/drive")

**Troubleshooting:**  
- If Colab becomes disconnected, Reconnect the runtime and rerun the mounting cell.
- If we receive an error such as `Mountpoint must not already contain files`, delete all the sub-folders under "/content/drive" from the Files panel before retrying. We need to delete these one by one starting from the innermost folders, until the last "drive" folder is deleted.

## 1.2 Install packages (only if needed)

In most cases, **Google Colab already includes the packages required** for this workflow. The most common missing dependency is **`netCDF4`** (NetCDF I/O support).

### Check what is already installed (Colab)
Before installing anything, you can inspect the current environment by running `!pip list`.

- If all required packages are present and only `netCDF4` is missing, install **only `netCDF4`**.
- If other required packages are missing from `!pip list`, install them **together with** `netCDF4` in the code cell below.

### Local Jupyter note
If you are running in a **local environment** (Jupyter / VS Code), assume all dependencies were installed when preparing the environment following the **main repository README**. In that case, you can skip this section.

Proceed to the code cell below only when installation is necessary.

In [ ]:
# In Google Colab, almost all packages already available, except netCDF4
!pip install netCDF4 cartopy

## 1.3 Creating Visualisations

This section loads the quality assessment NetCDF files produced by notebook 04 and creates diagnostic plots comparing correction quality across methods and spatial domains. Visualizations include CQI spatial maps, categorical classifications, method improvement maps, component breakdowns, and distribution analyses.

---

### Step 1: Environment Setup

Add the project root to the Python path so that the `src` package can be imported. Configuration is loaded from `config.yml` using `initialize_config()`.

**Note:** On Windows with conda, DLL directories are registered automatically to avoid `ImportError` issues.

In [ ]:
# Setup: Add project root to Python path
import os
import sys
import importlib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import logging

# Windows DLL fix for conda environments
if sys.platform == 'win32':
    _conda_prefix = os.environ.get('CONDA_PREFIX') or sys.prefix
    _dll_dirs = [
        os.path.join(_conda_prefix, 'Library', 'bin'),
        os.path.join(_conda_prefix, 'Library', 'lib'),
        os.path.join(_conda_prefix, 'Library', 'mingw-w64', 'bin'),
        os.path.join(_conda_prefix, 'bin'),
        _conda_prefix,
    ]
    for _d in _dll_dirs:
        if os.path.isdir(_d):
            try:
                os.add_dll_directory(_d)
            except OSError:
                pass
            if _d not in os.environ.get('PATH', ''):
                os.environ['PATH'] = _d + os.pathsep + os.environ.get('PATH', '') 
    del _conda_prefix, _dll_dirs, _d

# Add project root to path
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
project_root = '/content/drive/MyDrive/hybrid-bias-correction'
assert os.path.isfile(os.path.join(project_root, 'src', 'config.py')), f"Not found: {project_root}"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

for m in [k for k in sys.modules if k.startswith('src')]:
    del sys.modules[m]

import src.config as _cfg
importlib.reload(_cfg)
_cfg.initialize_config()

from src import config

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print(f"Project root:    {project_root}")
print(f"Quality path:    {config.quality_path_template}")
print(f"NetCDF engine:   {config.NETCDF_ENGINE}")

### Step 2: User Inputs

Select the month (1–12) and dekad (1, 2, or 3) for which to visualize quality assessment results.

In [ ]:
month_input = input('Enter the month (1-12): ').strip()
dekad_input = input('Enter the dekad (1, 2, or 3): ').strip()

try:
    month = int(month_input)
    assert 1 <= month <= 12
except (ValueError, AssertionError):
    raise SystemExit('Invalid month. Please provide a number from 1 to 12.')

try:
    dekad = int(dekad_input)
    assert dekad in (1, 2, 3)
except (ValueError, AssertionError):
    raise SystemExit('Invalid dekad. Must be 1, 2, or 3.')

month_str = f"{month:02d}"
dekad_str = '01' if dekad == 1 else ('11' if dekad == 2 else '21')

print(f'Selected: month={month}, dekad={dekad}')
print(f'Filename parts: month{month_str}_dekad{dekad_str}')

In [ ]:
"""
Figure output setup.

The quality_prefix variable (set in Step 3) is included in every
filename so that single-dekad and timeseries outputs never overwrite
each other.
"""
figures_dir = os.path.join(config.output_dir, 'figures')
os.makedirs(figures_dir, exist_ok=True)

titles_map = {'LS': 'LS', 'LSEQM': 'LSEQM', 'LSEQMDL': 'LSEQM+DL'}

def save_fig(fig, plot_type):
    """Save figure to the figures output directory."""
    fname = f"idn_cli_qa_viz_{quality_prefix}_{plot_type}_month{month_str}_dekad{dekad_str}.png"
    fpath = os.path.join(figures_dir, fname)
    fig.savefig(fpath, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"Saved: {fpath}")

print(f"Figures will be saved to: {figures_dir}")

### Step 3: Load QA Results

Load the quality assessment NetCDF files for all three bias correction methods.
Each file contains:
- `basic_statistical_quality` (0--1)
- `distribution_quality` (0--1)
- `temporal_quality` (0--1)
- `continuous_quality` (0--1) -- the CQI
- `categorical_quality` (1--4) -- Poor / Fair / Good / Excellent
- `confidence_level` (0--1)

In [ ]:
"""
Step 3: Load QA NetCDF files for each correction method.

Uses single-dekad files (qualitysd_) by default.
Change quality_prefix to 'qualityts' to load timeseries files instead.
"""
methods = {'LS': 'ls', 'LSEQM': 'lseqm', 'LSEQMDL': 'lseqmdl'}
ref_label = 'cpc'
quality_prefix = 'qualitysd'  # 'qualitysd' = single dekad, 'qualityts' = timeseries

quality_data = {}

for display_name, method_abbr in methods.items():
    quality_dir = config.quality_path_template.replace('{method}', method_abbr)
    test_label = f"imergl_{method_abbr}"
    fname = (
        f"idn_cli_{quality_prefix}_{ref_label}_{test_label}"
        f"_month{month_str}_dekad{dekad_str}.nc4"
    )
    fpath = os.path.join(quality_dir, fname)

    print(f"\n--- {display_name} ---")
    print(f"Path: {fpath}")

    if os.path.exists(fpath):
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE, decode_timedelta=False)
        quality_data[display_name] = ds
        print(f"Variables: {list(ds.data_vars)}")
        print(f"Dimensions: {dict(ds.sizes)}")
    else:
        print(f"WARNING: File not found -- skipping {display_name}")

if not quality_data:
    raise FileNotFoundError(
        "No QA output files found. Run notebook 04_qa_framework first."
    )

print(f"\nLoaded methods: {list(quality_data.keys())}")

### Step 4: CQI Spatial Maps

The **Continuous Quality Index (CQI)** is a weighted composite of basic statistical,
distribution, and temporal quality scores. Values range from 0 (poor) to 1 (excellent).

In [ ]:
"""
Step 4: CQI spatial maps for each method.
"""
n_methods = len(quality_data)
fig, axes = plt.subplots(
    1, n_methods, figsize=(5.5 * n_methods, 3.5),
    squeeze=False, constrained_layout=True
)
axes = axes.flatten()
im = None

for idx, (method_name, ds) in enumerate(quality_data.items()):
    ax = axes[idx]
    cqi = ds['continuous_quality']
    if 'time' in cqi.dims:
        cqi = cqi.isel(time=0)

    im = ax.pcolormesh(
        cqi.lon, cqi.lat, cqi.values,
        cmap='viridis', vmin=0, vmax=1, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(titles_map.get(method_name, method_name), fontsize=13)
    ax.set_xlabel('Longitude')
    if idx == 0:
        ax.set_ylabel('Latitude')
    else:
        ax.set_ylabel('')
    ax.set_aspect('equal')

if im is not None:
    fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
                 shrink=0.85, aspect=18, pad=0.02,
                 label='CQI (0 = poor, 1 = excellent)')

fig.suptitle(
    f'Continuous Quality Index (CQI) — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)

save_fig(fig, 'cqi_spatial')
plt.show()

### Step 5: Categorical Quality Maps

| Value | Category  | CQI Range |
|-------|-----------|-----------|
| 1     | Poor      | < 0.4     |
| 2     | Fair      | 0.4 -- 0.6|
| 3     | Good      | 0.6 -- 0.8|
| 4     | Excellent | >= 0.8    |

In [ ]:
"""
Step 5: Categorical quality maps (Poor / Fair / Good / Excellent).
"""
from matplotlib.colors import ListedColormap, BoundaryNorm

cat_colors = ['#d32f2f', '#ff9800', '#9ccc65', '#388e3c']
cat_cmap = ListedColormap(cat_colors)
cat_boundaries = [0.5, 1.5, 2.5, 3.5, 4.5]
cat_norm = BoundaryNorm(cat_boundaries, cat_cmap.N)

n_methods = len(quality_data)
fig, axes = plt.subplots(
    1, n_methods, figsize=(5.5 * n_methods, 3.5),
    squeeze=False, constrained_layout=True
)
axes = axes.flatten()
im = None

for idx, (method_name, ds) in enumerate(quality_data.items()):
    ax = axes[idx]
    cat = ds['categorical_quality']
    if 'time' in cat.dims:
        cat = cat.isel(time=0)
    cat_masked = cat.where(cat > 0)

    im = ax.pcolormesh(
        cat_masked.lon, cat_masked.lat, cat_masked.values,
        cmap=cat_cmap, norm=cat_norm, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(titles_map.get(method_name, method_name), fontsize=13)
    ax.set_xlabel('Longitude')
    if idx == 0:
        ax.set_ylabel('Latitude')
    else:
        ax.set_ylabel('')
    ax.set_aspect('equal')

if im is not None:
    cbar = fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
                        shrink=0.85, aspect=18, pad=0.02,
                        ticks=[1, 2, 3, 4])
    cbar.ax.set_yticklabels(['Poor', 'Fair', 'Good', 'Excellent'])
    cbar.set_label('Quality Category')

fig.suptitle(
    f'Categorical Quality Classification — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)

save_fig(fig, 'categorical_spatial')
plt.show()

### Step 6: Method Improvement Map

Quality gain (or loss) from the simplest method (LS) to the most advanced
(LSEQM+DL). Positive values (blue) = improvement, negative (red) = degradation.

In [ ]:
"""
Step 6: Method improvement map (LSEQMDL CQI minus LS CQI).
"""
if 'LS' in quality_data and 'LSEQMDL' in quality_data:
    cqi_ls = quality_data['LS']['continuous_quality']
    cqi_dl = quality_data['LSEQMDL']['continuous_quality']
    if 'time' in cqi_ls.dims:
        cqi_ls = cqi_ls.isel(time=0)
    if 'time' in cqi_dl.dims:
        cqi_dl = cqi_dl.isel(time=0)

    improvement = cqi_dl - cqi_ls

    fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
    im = ax.pcolormesh(
        improvement.lon, improvement.lat, improvement.values,
        cmap='RdBu_r', vmin=-0.5, vmax=0.5, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(
        f'CQI Improvement: LSEQM+DL minus LS — Month {month}, Dekad {dekad}',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    fig.colorbar(im, ax=ax, orientation='vertical',
                 shrink=0.75, aspect=20, pad=0.02,
                 label='CQI Difference (positive = improvement)')

    save_fig(fig, 'improvement')
    plt.show()

    # Summary
    imp_vals = improvement.values[~np.isnan(improvement.values)]
    n_total = len(imp_vals)
    if n_total > 0:
        n_improved = np.sum(imp_vals > 0.001)
        n_degraded = np.sum(imp_vals < -0.001)
        n_unchanged = n_total - n_improved - n_degraded
        print(f"Mean improvement:  {np.mean(imp_vals):.4f}")
        print(f"Median improvement: {np.median(imp_vals):.4f}")
        print(f"Pixels improved:   {n_improved} ({100*n_improved/n_total:.1f}%)")
        print(f"Pixels degraded:   {n_degraded} ({100*n_degraded/n_total:.1f}%)")
        print(f"Pixels unchanged:  {n_unchanged} ({100*n_unchanged/n_total:.1f}%)")
else:
    print('Both LS and LSEQMDL must be loaded for improvement map.')

### Step 7: Component Quality Maps

The CQI is composed of three weighted sub-scores (shown for LSEQM+DL):
- **Basic Statistical Quality** (weight 0.35): RB, RMSE, NSE
- **Distribution Quality** (weight 0.35): Percentile matching, variability, KS test
- **Temporal Quality** (weight 0.30): CSI, event timing, spell preservation

In [ ]:
"""
Step 7: Component quality maps (basic_statistical, distribution, temporal).
"""
comp_method = 'LSEQMDL' if 'LSEQMDL' in quality_data else list(quality_data.keys())[-1]
ds_comp = quality_data[comp_method]

components = [
    ('basic_statistical_quality', 'Basic Statistical'),
    ('distribution_quality', 'Distribution'),
    ('temporal_quality', 'Temporal'),
]

fig, axes = plt.subplots(
    1, 3, figsize=(16.5, 3.5),
    squeeze=False, constrained_layout=True
)
axes = axes.flatten()

for idx, (var_name, var_label) in enumerate(components):
    ax = axes[idx]
    data = ds_comp[var_name]
    if 'time' in data.dims:
        data = data.isel(time=0)

    im = ax.pcolormesh(
        data.lon, data.lat, data.values,
        cmap='viridis', vmin=0, vmax=1, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(var_label, fontsize=12)
    ax.set_xlabel('Longitude')
    if idx == 0:
        ax.set_ylabel('Latitude')
    else:
        ax.set_ylabel('')
    ax.set_aspect('equal')

fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
             shrink=0.85, aspect=18, pad=0.02,
             label='Score (0 = poor, 1 = excellent)')

fig.suptitle(
    f'Quality Components — {titles_map.get(comp_method, comp_method)} — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)

save_fig(fig, 'components')
plt.show()

### Step 8: Confidence Map

The **confidence level** indicates how reliable the quality assessment is at each
pixel. It combines metric consistency, distribution agreement (KS p-value),
and NSE reliability.

In [ ]:
"""
Step 8: Confidence map (LSEQMDL).
"""
conf_method = 'LSEQMDL' if 'LSEQMDL' in quality_data else list(quality_data.keys())[-1]
confidence = quality_data[conf_method]['confidence_level']
if 'time' in confidence.dims:
    confidence = confidence.isel(time=0)

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
im = ax.pcolormesh(
    confidence.lon, confidence.lat, confidence.values,
    cmap='YlOrRd_r', vmin=0, vmax=1, shading='auto'
)
ax.set_xlim(95, 141)
ax.set_ylim(-11, 6)
ax.set_title(
    f'Confidence Level — {titles_map.get(conf_method, conf_method)} — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')

fig.colorbar(im, ax=ax, orientation='vertical',
             shrink=0.75, aspect=20, pad=0.02,
             label='Confidence (0 = low, 1 = high)')

save_fig(fig, 'confidence')
plt.show()

conf_vals = confidence.values[~np.isnan(confidence.values)]
if len(conf_vals) > 0:
    print(f"Mean confidence:  {np.mean(conf_vals):.4f}")
    print(f"Min confidence:   {np.min(conf_vals):.4f}")
    print(f"Max confidence:   {np.max(conf_vals):.4f}")

### Step 9: CQI Distribution Analysis

- **Empirical CDF**: what fraction of pixels fall below each CQI value.
- **Histogram**: frequency distribution of CQI.

Vertical dashed lines mark the categorical quality boundaries (0.4, 0.6, 0.8).

In [ ]:
"""
Step 9: CQI distribution plots (CDF + histogram).
"""
method_colors = {'LS': '#1f77b4', 'LSEQM': '#ff7f0e', 'LSEQMDL': '#2ca02c'}

fig, (ax_cdf, ax_hist) = plt.subplots(1, 2, figsize=(14, 5))
cat_thresholds = [0.4, 0.6, 0.8]

for method_name, ds in quality_data.items():
    cqi = ds['continuous_quality']
    if 'time' in cqi.dims:
        cqi = cqi.isel(time=0)
    vals = cqi.values.flatten()
    vals = vals[~np.isnan(vals)]

    color = method_colors.get(method_name, None)
    label = titles_map.get(method_name, method_name)

    # Empirical CDF
    sorted_vals = np.sort(vals)
    ecdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax_cdf.plot(sorted_vals, ecdf, label=label, color=color, linewidth=1.5)

    # Histogram
    ax_hist.hist(vals, bins=20, alpha=0.45, label=label, color=color,
                 edgecolor='white', linewidth=0.5, range=(0, 1))

    print(f"Median CQI ({label}): {np.median(vals):.4f}")

# Category boundary lines
for thresh in cat_thresholds:
    ax_cdf.axvline(thresh, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax_hist.axvline(thresh, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

ax_cdf.set_xlabel('CQI')
ax_cdf.set_ylabel('Cumulative Fraction')
ax_cdf.set_title('Empirical CDF of CQI')
ax_cdf.set_xlim(0, 1)
ax_cdf.set_ylim(0, 1)
ax_cdf.legend()
ax_cdf.grid(True, alpha=0.3)

ax_hist.set_xlabel('CQI')
ax_hist.set_ylabel('Pixel Count')
ax_hist.set_title('Histogram of CQI')
ax_hist.set_xlim(0, 1)
ax_hist.legend()
ax_hist.grid(True, alpha=0.3)

fig.suptitle(
    f'CQI Distribution Analysis — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
save_fig(fig, 'cqi_distribution')
plt.show()

### Step 10: Quality Category Summary

Grouped bar chart and percentage table showing how many pixels fall into each
quality category for each correction method.

In [ ]:
"""
Step 10: Quality category summary (bar chart + table).
"""
cat_names = ['Poor', 'Fair', 'Good', 'Excellent']
cat_values = [1, 2, 3, 4]
method_bar_colors = {'LS': '#1f77b4', 'LSEQM': '#ff7f0e', 'LSEQMDL': '#2ca02c'}

# Compute counts per category per method
summary = {}
for method_name, ds in quality_data.items():
    cat = ds['categorical_quality']
    if 'time' in cat.dims:
        cat = cat.isel(time=0)
    vals = cat.values.flatten()
    vals = vals[(vals > 0) & ~np.isnan(vals)]
    total = len(vals)
    counts = [int(np.sum(vals == cv)) for cv in cat_values]
    summary[method_name] = {'counts': counts, 'total': total}

# Grouped bar chart
x = np.arange(len(cat_names))
width = 0.25
n = len(quality_data)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (method_name, info) in enumerate(summary.items()):
    offset = (i - (n - 1) / 2) * width
    pcts = [100 * c / info['total'] if info['total'] > 0 else 0
            for c in info['counts']]
    label = titles_map.get(method_name, method_name)
    ax.bar(x + offset, pcts, width, label=label,
           color=method_bar_colors.get(method_name, f'C{i}'),
           edgecolor='white', linewidth=0.5)

ax.set_xlabel('Quality Category')
ax.set_ylabel('Percentage of Pixels (%)')
ax.set_title(
    f'Quality Category Distribution — Month {month}, Dekad {dekad}',
    fontsize=13, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels(cat_names)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig, 'category_summary')
plt.show()

# Percentage table
print(f"\n{'Method':<12}  {'Poor':>8}  {'Fair':>8}  {'Good':>8}  {'Excellent':>10}  {'Total':>10}")
print('-' * 68)
for method_name, info in summary.items():
    total = info['total']
    pcts = [100 * c / total if total > 0 else 0 for c in info['counts']]
    label = titles_map.get(method_name, method_name)
    print(f"{label:<12}  {pcts[0]:>7.1f}%  {pcts[1]:>7.1f}%  {pcts[2]:>7.1f}%"
          f"  {pcts[3]:>9.1f}%  {total:>10,}")

### Step 11: Component Score Box Plots

Box plots comparing the distribution of each quality component across all
methods. This identifies which aspect benefits most from progressive refinement.

In [ ]:
"""
Step 11: Component score box plots.
"""
try:
    import seaborn as sns
    use_seaborn = True
except ImportError:
    use_seaborn = False
    print('seaborn not available; using matplotlib box plots.')

component_vars = [
    ('basic_statistical_quality', 'Basic Statistical'),
    ('distribution_quality', 'Distribution'),
    ('temporal_quality', 'Temporal'),
    ('continuous_quality', 'CQI'),
]

# Build long-form DataFrame
records = []
for method_name, ds in quality_data.items():
    label = titles_map.get(method_name, method_name)
    for var_name, var_label in component_vars:
        data = ds[var_name]
        if 'time' in data.dims:
            data = data.isel(time=0)
        vals = data.values.flatten()
        vals = vals[~np.isnan(vals)]
        for v in vals:
            records.append({'Method': label, 'Component': var_label, 'Score': float(v)})

df_box = pd.DataFrame(records)

if use_seaborn and len(df_box) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df_box, x='Component', y='Score', hue='Method',
                ax=ax, palette='Set2', fliersize=1, linewidth=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(
        f'Quality Component Scores by Method — Month {month}, Dekad {dekad}',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Score (0-1)')
    ax.set_xlabel('Quality Component')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title='Method')
    plt.tight_layout()
    save_fig(fig, 'component_boxplots')
    plt.show()

elif len(df_box) > 0:
    method_order = [titles_map.get(m, m) for m in quality_data.keys()]
    n_comp = len(component_vars)
    fig, axes = plt.subplots(1, n_comp, figsize=(4 * n_comp, 6), squeeze=False)
    axes = axes.flatten()
    for c_idx, (var_name, var_label) in enumerate(component_vars):
        ax = axes[c_idx]
        bp_data = [df_box[(df_box['Component'] == var_label) &
                          (df_box['Method'] == m)]['Score'].values
                   for m in method_order]
        ax.boxplot(bp_data, labels=method_order, patch_artist=True)
        ax.set_ylim(0, 1)
        ax.set_title(var_label)
        ax.set_ylabel('Score')
        ax.grid(True, axis='y', alpha=0.3)
    fig.suptitle(
        f'Quality Component Scores by Method — Month {month}, Dekad {dekad}',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    save_fig(fig, 'component_boxplots')
    plt.show()

else:
    print('No data available for box plots.')

---


### Step 12: Batch Export — All Months and Dekads

Run this cell to generate **all 8 plot types** for every month×dekad combination
(12 months × 3 dekads = 36 periods, up to **288 figures**). Figures are saved
silently to the `figures/` directory without displaying in the notebook.

Only periods that have matching QA NetCDF files are processed; missing periods
are skipped automatically.

**Prerequisites — run these cells first (skip Steps 4–11):**

| Cell | Purpose |
|------|---------|
| Step 1 | Environment setup, imports, `config` |
| Step 2 | User inputs — sets `month_str`, `dekad_str` (values are overridden by the batch loop, but the cell initialises the variables) |
| Figure output setup | Defines `figures_dir`, `titles_map`, and `save_fig()` |
| Step 3 | Sets `quality_prefix` (`qualitysd` or `qualityts`) and validates paths |

In [ ]:
"""
Step 12: Batch export — loop all 36 month×dekad combinations.

Reuses quality_prefix, figures_dir, titles_map, save_fig, and config
from the cells above. Overwrites the global month/dekad/month_str/dekad_str
variables on each iteration so save_fig picks up the right filename.
"""
from matplotlib.colors import ListedColormap, BoundaryNorm

try:
    import seaborn as sns
    _batch_seaborn = True
except ImportError:
    _batch_seaborn = False

methods = {'LS': 'ls', 'LSEQM': 'lseqm', 'LSEQMDL': 'lseqmdl'}
ref_label = 'cpc'
method_bar_colors = {'LS': '#1f77b4', 'LSEQM': '#ff7f0e', 'LSEQMDL': '#2ca02c'}
method_line_colors = method_bar_colors.copy()

cat_colors = ['#d32f2f', '#ff9800', '#9ccc65', '#388e3c']
cat_cmap = ListedColormap(cat_colors)
cat_boundaries = [0.5, 1.5, 2.5, 3.5, 4.5]
cat_norm = BoundaryNorm(cat_boundaries, cat_cmap.N)
cat_thresholds = [0.4, 0.6, 0.8]
cat_names = ['Poor', 'Fair', 'Good', 'Excellent']
cat_values_list = [1, 2, 3, 4]

component_vars = [
    ('basic_statistical_quality', 'Basic Statistical'),
    ('distribution_quality', 'Distribution'),
    ('temporal_quality', 'Temporal'),
    ('continuous_quality', 'CQI'),
]

n_saved = 0
n_skipped = 0

for _m in range(1, 13):
    for _d in [1, 2, 3]:
        # Update globals so save_fig resolves the right filename
        month = _m
        dekad = _d
        month_str = f"{_m:02d}"
        dekad_str = '01' if _d == 1 else ('11' if _d == 2 else '21')
        tag = f"month {_m:02d} dekad {_d}"

        # --- Load ---
        quality_data = {}
        for display_name, method_abbr in methods.items():
            quality_dir = config.quality_path_template.replace('{method}', method_abbr)
            test_label = f"imergl_{method_abbr}"
            fname = (
                f"idn_cli_{quality_prefix}_{ref_label}_{test_label}"
                f"_month{month_str}_dekad{dekad_str}.nc4"
            )
            fpath = os.path.join(quality_dir, fname)
            if os.path.exists(fpath):
                quality_data[display_name] = xr.open_dataset(
                    fpath, engine=config.NETCDF_ENGINE, decode_timedelta=False
                )

        if not quality_data:
            n_skipped += 1
            continue

        print(f"▸ {tag} — {len(quality_data)} method(s)  ", end="")

        # --- 1. CQI spatial (3-panel) ---
        n_meth = len(quality_data)
        fig, axes = plt.subplots(1, n_meth, figsize=(5.5 * n_meth, 3.5),
                                 squeeze=False, constrained_layout=True)
        axes = axes.flatten()
        im = None
        for idx, (mn, ds) in enumerate(quality_data.items()):
            ax = axes[idx]
            cqi = ds['continuous_quality']
            if 'time' in cqi.dims: cqi = cqi.isel(time=0)
            im = ax.pcolormesh(cqi.lon, cqi.lat, cqi.values,
                               cmap='viridis', vmin=0, vmax=1, shading='auto')
            ax.set_xlim(95, 141); ax.set_ylim(-11, 6)
            ax.set_title(titles_map.get(mn, mn), fontsize=13)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude' if idx == 0 else '')
            ax.set_aspect('equal')
        if im:
            fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
                         shrink=0.85, aspect=18, pad=0.02,
                         label='CQI (0 = poor, 1 = excellent)')
        fig.suptitle(f'CQI — Month {month}, Dekad {dekad}', fontsize=13, fontweight='bold')
        save_fig(fig, 'cqi_spatial'); plt.close(fig)

        # --- 2. Categorical quality (3-panel) ---
        fig, axes = plt.subplots(1, n_meth, figsize=(5.5 * n_meth, 3.5),
                                 squeeze=False, constrained_layout=True)
        axes = axes.flatten()
        im = None
        for idx, (mn, ds) in enumerate(quality_data.items()):
            ax = axes[idx]
            cat = ds['categorical_quality']
            if 'time' in cat.dims: cat = cat.isel(time=0)
            cat_masked = cat.where(cat > 0)
            im = ax.pcolormesh(cat_masked.lon, cat_masked.lat, cat_masked.values,
                               cmap=cat_cmap, norm=cat_norm, shading='auto')
            ax.set_xlim(95, 141); ax.set_ylim(-11, 6)
            ax.set_title(titles_map.get(mn, mn), fontsize=13)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude' if idx == 0 else '')
            ax.set_aspect('equal')
        if im:
            cbar = fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
                                shrink=0.85, aspect=18, pad=0.02, ticks=[1, 2, 3, 4])
            cbar.ax.set_yticklabels(['Poor', 'Fair', 'Good', 'Excellent'])
            cbar.set_label('Quality Category')
        fig.suptitle(f'Categorical Quality — Month {month}, Dekad {dekad}',
                     fontsize=13, fontweight='bold')
        save_fig(fig, 'categorical_spatial'); plt.close(fig)

        # --- 3. Improvement map (single panel) ---
        if 'LS' in quality_data and 'LSEQMDL' in quality_data:
            cqi_ls = quality_data['LS']['continuous_quality']
            cqi_dl = quality_data['LSEQMDL']['continuous_quality']
            if 'time' in cqi_ls.dims: cqi_ls = cqi_ls.isel(time=0)
            if 'time' in cqi_dl.dims: cqi_dl = cqi_dl.isel(time=0)
            improvement = cqi_dl - cqi_ls
            fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
            im = ax.pcolormesh(improvement.lon, improvement.lat, improvement.values,
                               cmap='RdBu_r', vmin=-0.5, vmax=0.5, shading='auto')
            ax.set_xlim(95, 141); ax.set_ylim(-11, 6)
            ax.set_title(f'CQI Improvement — Month {month}, Dekad {dekad}',
                         fontsize=13, fontweight='bold')
            ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
            ax.set_aspect('equal')
            fig.colorbar(im, ax=ax, orientation='vertical',
                         shrink=0.75, aspect=20, pad=0.02,
                         label='CQI Difference')
            save_fig(fig, 'improvement'); plt.close(fig)

        # --- 4. Component quality (3-panel) ---
        comp_method = 'LSEQMDL' if 'LSEQMDL' in quality_data else list(quality_data.keys())[-1]
        ds_comp = quality_data[comp_method]
        components = [('basic_statistical_quality', 'Basic Statistical'),
                      ('distribution_quality', 'Distribution'),
                      ('temporal_quality', 'Temporal')]
        fig, axes = plt.subplots(1, 3, figsize=(16.5, 3.5),
                                 squeeze=False, constrained_layout=True)
        axes = axes.flatten()
        for idx, (vn, vl) in enumerate(components):
            ax = axes[idx]
            data = ds_comp[vn]
            if 'time' in data.dims: data = data.isel(time=0)
            im = ax.pcolormesh(data.lon, data.lat, data.values,
                               cmap='viridis', vmin=0, vmax=1, shading='auto')
            ax.set_xlim(95, 141); ax.set_ylim(-11, 6)
            ax.set_title(vl, fontsize=12)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude' if idx == 0 else '')
            ax.set_aspect('equal')
        fig.colorbar(im, ax=axes.tolist(), orientation='vertical',
                     shrink=0.85, aspect=18, pad=0.02,
                     label='Score (0 = poor, 1 = excellent)')
        fig.suptitle(f'Components — {titles_map.get(comp_method, comp_method)} — Month {month}, Dekad {dekad}',
                     fontsize=13, fontweight='bold')
        save_fig(fig, 'components'); plt.close(fig)

        # --- 5. Confidence map (single panel) ---
        conf_method = comp_method
        confidence = quality_data[conf_method]['confidence_level']
        if 'time' in confidence.dims: confidence = confidence.isel(time=0)
        fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
        im = ax.pcolormesh(confidence.lon, confidence.lat, confidence.values,
                           cmap='YlOrRd_r', vmin=0, vmax=1, shading='auto')
        ax.set_xlim(95, 141); ax.set_ylim(-11, 6)
        ax.set_title(f'Confidence — {titles_map.get(conf_method, conf_method)} — Month {month}, Dekad {dekad}',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        ax.set_aspect('equal')
        fig.colorbar(im, ax=ax, orientation='vertical',
                     shrink=0.75, aspect=20, pad=0.02,
                     label='Confidence (0 = low, 1 = high)')
        save_fig(fig, 'confidence'); plt.close(fig)

        # --- 6. CQI distribution (CDF + histogram) ---
        fig, (ax_cdf, ax_hist) = plt.subplots(1, 2, figsize=(14, 5))
        for mn, ds in quality_data.items():
            cqi = ds['continuous_quality']
            if 'time' in cqi.dims: cqi = cqi.isel(time=0)
            vals = cqi.values.flatten()
            vals = vals[~np.isnan(vals)]
            color = method_line_colors.get(mn, None)
            label = titles_map.get(mn, mn)
            sorted_vals = np.sort(vals)
            ecdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
            ax_cdf.plot(sorted_vals, ecdf, label=label, color=color, linewidth=1.5)
            ax_hist.hist(vals, bins=20, alpha=0.45, label=label, color=color,
                         edgecolor='white', linewidth=0.5, range=(0, 1))
        for th in cat_thresholds:
            ax_cdf.axvline(th, color='gray', ls='--', alpha=0.5, lw=0.8)
            ax_hist.axvline(th, color='gray', ls='--', alpha=0.5, lw=0.8)
        ax_cdf.set(xlabel='CQI', ylabel='Cumulative Fraction', title='CDF', xlim=(0,1), ylim=(0,1))
        ax_cdf.legend(); ax_cdf.grid(True, alpha=0.3)
        ax_hist.set(xlabel='CQI', ylabel='Pixel Count', title='Histogram', xlim=(0,1))
        ax_hist.legend(); ax_hist.grid(True, alpha=0.3)
        fig.suptitle(f'CQI Distribution — Month {month}, Dekad {dekad}',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        save_fig(fig, 'cqi_distribution'); plt.close(fig)

        # --- 7. Category summary (bar chart) ---
        summary = {}
        for mn, ds in quality_data.items():
            cat = ds['categorical_quality']
            if 'time' in cat.dims: cat = cat.isel(time=0)
            vals = cat.values.flatten()
            vals = vals[(vals > 0) & ~np.isnan(vals)]
            total = len(vals)
            counts = [int(np.sum(vals == cv)) for cv in cat_values_list]
            summary[mn] = {'counts': counts, 'total': total}
        x = np.arange(len(cat_names))
        width = 0.25; n = len(quality_data)
        fig, ax = plt.subplots(figsize=(10, 5))
        for i, (mn, info) in enumerate(summary.items()):
            offset = (i - (n - 1) / 2) * width
            pcts = [100 * c / info['total'] if info['total'] > 0 else 0 for c in info['counts']]
            ax.bar(x + offset, pcts, width, label=titles_map.get(mn, mn),
                   color=method_bar_colors.get(mn, f'C{i}'), edgecolor='white', linewidth=0.5)
        ax.set(xlabel='Quality Category', ylabel='Percentage of Pixels (%)')
        ax.set_title(f'Category Distribution — Month {month}, Dekad {dekad}',
                     fontsize=13, fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(cat_names)
        ax.legend(); ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        save_fig(fig, 'category_summary'); plt.close(fig)

        # --- 8. Component box plots ---
        records = []
        for mn, ds in quality_data.items():
            label = titles_map.get(mn, mn)
            for vn, vl in component_vars:
                data = ds[vn]
                if 'time' in data.dims: data = data.isel(time=0)
                vals = data.values.flatten()
                vals = vals[~np.isnan(vals)]
                for v in vals:
                    records.append({'Method': label, 'Component': vl, 'Score': float(v)})
        df_box = pd.DataFrame(records)
        if _batch_seaborn and len(df_box) > 0:
            fig, ax = plt.subplots(figsize=(12, 6))
            sns.boxplot(data=df_box, x='Component', y='Score', hue='Method',
                        ax=ax, palette='Set2', fliersize=1, linewidth=0.8)
            ax.set_ylim(0, 1)
            ax.set_title(f'Component Scores — Month {month}, Dekad {dekad}',
                         fontsize=13, fontweight='bold')
            ax.set_ylabel('Score (0-1)'); ax.set_xlabel('Quality Component')
            ax.grid(True, axis='y', alpha=0.3); ax.legend(title='Method')
            plt.tight_layout()
            save_fig(fig, 'component_boxplots'); plt.close(fig)
        elif len(df_box) > 0:
            method_order = [titles_map.get(m, m) for m in quality_data.keys()]
            n_comp = len(component_vars)
            fig, axes = plt.subplots(1, n_comp, figsize=(4 * n_comp, 6), squeeze=False)
            axes = axes.flatten()
            for c_idx, (vn, vl) in enumerate(component_vars):
                ax = axes[c_idx]
                bp_data = [df_box[(df_box['Component'] == vl) &
                                  (df_box['Method'] == m)]['Score'].values for m in method_order]
                ax.boxplot(bp_data, labels=method_order, patch_artist=True)
                ax.set_ylim(0, 1); ax.set_title(vl); ax.set_ylabel('Score')
                ax.grid(True, axis='y', alpha=0.3)
            fig.suptitle(f'Component Scores — Month {month}, Dekad {dekad}',
                         fontsize=13, fontweight='bold')
            plt.tight_layout()
            save_fig(fig, 'component_boxplots'); plt.close(fig)

        # Close loaded datasets
        for ds in quality_data.values():
            ds.close()

        n_saved += 1
        print("✓")

print(f"\nBatch complete: {n_saved} periods exported, {n_skipped} skipped (files not found).")
print(f"Figures directory: {figures_dir}")